# pipe_nuevo — 04 Explorar features: ejemplos concretos para chequear que tengan sentido

No arma nada nuevo: lee el parquet final que dejo `03_Escalado.ipynb` y mira con
lupa las features NUEVAS de `02_FE.ipynb` (edad de producto vs. edad del par,
recencia de compra, peso acumulado, vecinos por correlacion, escalado) sobre
ejemplos concretos -- que producto es, que categoria, como se ve en el tiempo --
en vez de solo confiar en el control de leakage automatico.

Cada seccion tiene: una tabla chica y legible (con `cat1/cat2/cat3/brand`, no
solo `product_id`), un grafico, y para vecinos ademas una matriz de correlacion
para ver visualmente quien se parece a quien.


In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    import os
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
RUTA_FE = BUCKET / "datasets_fe"

SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
GRILLA, EJE_C, FONDO = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO,
    "axes.edgecolor": EJE_C, "axes.labelcolor": TINTA2,
    "text.color": TINTA, "xtick.color": MUDO, "ytick.color": MUDO,
    "grid.color": GRILLA, "grid.linewidth": .8,
    "axes.grid": True, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
    "legend.frameon": False,
})


def limpiar(ax, titulo=None, y=None, x=None):
    if titulo:
        ax.set_title(titulo, color=TINTA, loc="left", pad=10)
    if y:
        ax.set_ylabel(y)
    if x:
        ax.set_xlabel(x)
    ax.grid(axis="x", visible=False)
    return ax


print(f"BUCKET  : {BUCKET}")
print(f"datasets: {RUTA_FE}")


### Inventario: que parquets de `03_Escalado` hay disponibles


In [ ]:
disponibles = sorted(RUTA_FE.glob("preprocesado_*_pipeNuevo.parquet"))
print(f"{len(disponibles)} parquet(s) de pipe_nuevo en {RUTA_FE.name}/:")
for p in disponibles:
    print(f"  - {p.name}   ({p.stat().st_size/1e6:.0f} MB)")
if not disponibles:
    raise FileNotFoundError(
        f"No hay preprocesado_*_pipeNuevo.parquet en {RUTA_FE}. "
        f"Corre 01_Preprocesamiento -> 02_FE -> 03_Escalado primero."
    )

PARAM = {
    # Nombre exacto de la lista de arriba. None = el modificado mas reciente.
    'archivo': None,

    # Cuantos productos/pares mostrar en cada tabla y grafico de ejemplo.
    'n_ejemplos': 6,
    # Top-N productos por volumen para la matriz de correlacion (mas de ~30
    # se vuelve ilegible en el heatmap).
    'n_matriz': 20,

    'semilla': 102191,
}

nombre = PARAM['archivo'] or max(disponibles, key=lambda p: p.stat().st_mtime).name
path_pre = RUTA_FE / nombre
print(f"\nUsando: {nombre}")


In [ ]:
df = pl.read_parquet(path_pre)
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))

ES_PC = "customer_id" in df.columns
KEYS = ["product_id", "customer_id"] if ES_PC else ["product_id"]
CATS_ID = [c for c in ("cat1", "cat2", "cat3", "brand", "sku_size") if c in df.columns]

# si el parquet no trae las categorias pegadas (deberian venir de 02_FE), se pegan aca
if not CATS_ID:
    df = df.join(prod.select(["product_id"] + [c for c in ("cat1", "cat2", "cat3", "brand", "sku_size")
                                                if c in prod.columns]),
                 on="product_id", how="left")
    CATS_ID = [c for c in ("cat1", "cat2", "cat3", "brand", "sku_size") if c in df.columns]

print(f"granularidad: {'cliente-producto' if ES_PC else 'producto'}   (claves: {KEYS})")
print(f"{df.height:,} filas x {df.width} columnas")
print(f"categorias disponibles: {CATS_ID}")
print(f"rango de periodos: {df['periodo'].min()} -> {df['periodo'].max()}")

_features_json = RUTA_FE / nombre.replace(".parquet", "_features.json")
if _features_json.exists():
    with open(_features_json, encoding="utf-8") as f:
        _meta = json.load(f)
    print(f"\nfamilias de features (de {_features_json.name}):")
    for k, v in _meta.get("familias", {}).items():
        print(f"  {k:24s} {len(v):3d}")


### Tarjeta de producto

No hay un "nombre" de producto en `tb_productos.txt` -- la identidad de un
producto ES su jerarquia comercial (`cat1/cat2/cat3/brand/sku_size`). La
tarjeta junta eso con lo que calculamos: cuando nacio, cuanto vende, y quienes
son sus vecinos (con SUS categorias tambien, para poder juzgar a ojo si el
match tiene sentido).


In [ ]:
# vecinos: se buscan en el _vecinos.json que escribe 02_FE (mismo prefijo de
# granularidad que el parquet elegido). Si no esta (por ej. porque 02_FE se
# corrio con otros parametros), la tarjeta sigue funcionando sin esa parte.
_grp = "grpClienteProducto" if ES_PC else "grpProducto"
_vec_files = sorted(RUTA_FE.glob(f"features_sin_escalar_{_grp}*_vecinos.json"))
VECINOS = {}
if _vec_files:
    _vf = max(_vec_files, key=lambda p: p.stat().st_mtime)
    with open(_vf, encoding="utf-8") as f:
        VECINOS = json.load(f)
    print(f"vecinos leidos de: {_vf.name}  ({len(VECINOS)} productos)")
else:
    print("no encontre *_vecinos.json -- la tarjeta se muestra sin esa seccion")

CAT_POR_PRODUCTO = {r["product_id"]: r for r in
                    prod.select(["product_id"] + CATS_ID).to_dicts()} if CATS_ID else {}


def describir_producto(pid):
    c = CAT_POR_PRODUCTO.get(pid, {})
    partes = [f"{k}={c.get(k)}" for k in CATS_ID if k in c]
    return f"producto {pid} (" + ", ".join(partes) + ")" if partes else f"producto {pid}"


def tarjeta(product_id):
    sub = df.filter(pl.col("product_id") == product_id).sort("periodo")
    if sub.height == 0:
        print(f"no hay filas para product_id={product_id}")
        return
    print("=" * 78)
    print(describir_producto(product_id))
    print("=" * 78)
    ultimo = sub.tail(1)
    if "edad_producto" in sub.columns:
        nace_en = sub.filter(pl.col("edad_producto") == 0)["periodo"]
        print(f"  nace (edad_producto=0) en: {nace_en[0] if len(nace_en) else '?'}")
    print(f"  tn0 total en el parquet: {sub['tn0'].sum():,.1f}")
    print(f"  ultimo periodo: {ultimo['periodo'][0]}   tn0: {ultimo['tn0'][0]:.2f}")
    if "peso_producto_acum" in sub.columns:
        print(f"  peso_producto_acum al final: {ultimo['peso_producto_acum'][0]:.4f}")
    if "meses_sin_compra" in sub.columns and ES_PC:
        print(f"  meses_sin_compra (ultimo par de esta fila): {ultimo['meses_sin_compra'][0]}")

    v = VECINOS.get(str(product_id))
    if v:
        print("\n  sustitutos (top correlacion NEGATIVA):")
        for vid in v.get("sustitutos", []):
            print(f"    - {describir_producto(vid)}")
        print("  complementarios (top correlacion POSITIVA):")
        for vid in v.get("complementarios", []):
            print(f"    - {describir_producto(vid)}")


TOP_VOLUMEN = (df.group_by("product_id").agg(pl.col("tn0").sum().alias("tn_total"))
                 .sort("tn_total", descending=True)["product_id"].to_list())

for pid in TOP_VOLUMEN[:PARAM['n_ejemplos']]:
    tarjeta(pid)


### Edad: producto vs. par cliente-producto

Si `edad_cliente_producto` == `edad_producto` en TODAS las filas, algo esta mal (significaria que ningun cliente empezo a comprar un producto que ya existia -- osea que el par siempre nace junto con el producto).


In [ ]:
if "edad_producto" in df.columns and "edad_cliente_producto" in df.columns:
    dif = df.filter(pl.col("edad_producto") != pl.col("edad_cliente_producto"))
    print(f"filas donde el par empezo DESPUES que el producto: {dif.height:,} "
          f"de {df.height:,} ({100*dif.height/df.height:.0f}%)")
    if ES_PC and dif.height:
        ej = dif.sort("edad_producto", descending=True).head(PARAM['n_ejemplos'])
        print("\nejemplos (producto viejo, cliente nuevo en el):")
        print(ej.select(["product_id", "customer_id", "periodo",
                         "edad_producto", "edad_cliente_producto"] + CATS_ID[:2]))

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].hist(df["edad_producto"].drop_nulls(), bins=30, color=SERIE[0])
    limpiar(axes[0], "distribucion de edad_producto (todas las filas)", "filas", "meses desde el lanzamiento")
    if ES_PC:
        muestra = df.sample(n=min(20_000, df.height), seed=PARAM['semilla'])
        axes[1].scatter(muestra["edad_producto"], muestra["edad_cliente_producto"],
                        s=4, alpha=.25, color=SERIE[1])
        lim = max(muestra["edad_producto"].max(), muestra["edad_cliente_producto"].max())
        axes[1].plot([0, lim], [0, lim], color=MUDO, linewidth=.8, linestyle=":")
        limpiar(axes[1], "edad_producto vs edad_cliente_producto\n(la diagonal = el cliente compro desde el dia 1)",
                "edad_cliente_producto", "edad_producto")
    else:
        axes[1].axis("off")
    fig.tight_layout()
    plt.show()


### Recencia de compra (`meses_sin_compra`)

Ejemplo: para un par con ventas intermitentes, `meses_sin_compra` tiene que volver a 0 justo despues de cada compra y crecer 1 a 1 mientras no compra.


In [ ]:
if "meses_sin_compra" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    axes[0].hist(df["meses_sin_compra"].drop_nulls(), bins=30, color=SERIE[2])
    limpiar(axes[0], "distribucion de meses_sin_compra", "filas", "meses desde la ultima compra")

    # se busca un par intermitente de verdad: con ventas Y con huecos
    cand = (df.group_by(KEYS)
              .agg(pl.col("tn0").filter(pl.col("tn0") > 0).count().alias("n_compras"),
                   pl.len().alias("n_filas"))
              .filter((pl.col("n_compras") >= 3) & (pl.col("n_compras") < pl.col("n_filas") * 0.6))
              .sort("n_filas", descending=True))
    if cand.height:
        ejemplo = {k: cand[k][0] for k in KEYS}
        serie = df.filter(pl.all_horizontal([pl.col(k) == v for k, v in ejemplo.items()])).sort("periodo")
        ax = axes[1]
        ax2 = ax.twinx()
        ax.bar(range(serie.height), serie["tn0"], color=SERIE[0], alpha=.7, width=.6)
        ax2.plot(range(serie.height), serie["meses_sin_compra"], color=TINTA, linewidth=1.6,
                 marker="o", markersize=3)
        limpiar(ax, f"{describir_producto(ejemplo['product_id'])}\ntn0 (barras) vs meses_sin_compra (linea)",
                "tn0", "periodo (indice)")
        ax2.set_ylabel("meses_sin_compra")
        ax2.grid(False)
    fig.tight_layout()
    plt.show()


### Peso acumulado en las ventas totales

Los productos/clientes de mayor `peso_*_acum` al final del panel tienen que ser, a ojo, los de mayor volumen historico -- si no coincide, hay un bug en el cumsum.


In [ ]:
if "peso_producto_acum" in df.columns:
    ultimo_periodo = df["periodo"].max()
    top_peso = (df.filter(pl.col("periodo") == ultimo_periodo)
                  .unique(subset=["product_id"])
                  .sort("peso_producto_acum", descending=True)
                  .head(PARAM['n_ejemplos']))
    print(f"top {PARAM['n_ejemplos']} productos por peso_producto_acum en {ultimo_periodo}:")
    print(top_peso.select(["product_id", "peso_producto_acum"] + CATS_ID[:3]))

    fig, ax = plt.subplots(figsize=(8, 3.8))
    for j, pid in enumerate(top_peso["product_id"].to_list()):
        serie = (df.filter(pl.col("product_id") == pid)
                   .unique(subset=["product_id", "periodo"])
                   .sort("periodo"))
        ax.plot(serie["periodo"].cast(str), serie["peso_producto_acum"],
                color=SERIE[j % len(SERIE)], linewidth=1.6, label=describir_producto(pid)[:30])
    ax.set_xticks(ax.get_xticks()[::max(1, len(ax.get_xticks()) // 8)])
    limpiar(ax, "peso_producto_acum en el tiempo -- top productos", "peso acumulado", "periodo")
    ax.legend(fontsize=6.5, ncols=2)
    fig.tight_layout()
    plt.show()


### Vecinos por correlacion: matriz y ejemplo en el tiempo

Matriz de correlacion producto x producto (Top-N por volumen, recalculada aca
mismo desde el parquet para poder mirarla) -- rojo: se mueven juntos
(candidatos a complementarios), azul: se mueven al reves (candidatos a
sustitutos). Abajo, un producto de ejemplo con su venta real contra el
promedio de sus sustitutos/complementarios: si la feature tiene sentido, la
linea de sustitutos deberia moverse mas seguido al reves de la propia.


In [ ]:
tot_prod = (df.group_by(["product_id", "periodo"]).agg(pl.col("tn0").sum().alias("tn_prod"))
              if ES_PC else
              df.select(["product_id", "periodo", pl.col("tn0").alias("tn_prod")]))

top_n = (tot_prod.group_by("product_id").agg(pl.col("tn_prod").sum().alias("tn_total"))
                 .sort("tn_total", descending=True).head(PARAM['n_matriz'])["product_id"].to_list())

wide = (tot_prod.filter(pl.col("product_id").is_in(top_n))
                .pivot(on="product_id", index="periodo", values="tn_prod")
                .sort("periodo")
                .drop("periodo"))
corr = wide.to_pandas().corr(method="spearman")   # igual metodo que 02_FE
orden = [str(p) for p in top_n if str(p) in corr.columns]
corr = corr.loc[orden, orden]

fig, ax = plt.subplots(figsize=(1 + .5 * len(orden), 1 + .5 * len(orden)))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(orden)))
ax.set_xticklabels(orden, rotation=90, fontsize=6.5)
ax.set_yticks(range(len(orden)))
ax.set_yticklabels(orden, fontsize=6.5)
ax.grid(False)
limpiar(ax, f"correlacion de tn entre los {len(orden)} productos de mayor volumen", None, None)
fig.colorbar(im, ax=ax, label="correlacion", fraction=.03)
fig.tight_layout()
plt.show()

if "tn_sustitutos_prom" in df.columns:
    pid_ej = top_n[0]
    serie = (df.filter(pl.col("product_id") == pid_ej)
               .unique(subset=["product_id", "periodo"])
               .sort("periodo"))
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(serie["periodo"].cast(str), serie["tn_prod"] if "tn_prod" in serie.columns else serie["tn0"],
            color=TINTA, linewidth=1.8, label=f"{describir_producto(pid_ej)[:30]} (propio)")
    ax.plot(serie["periodo"].cast(str), serie["tn_sustitutos_prom"], color=SERIE[3],
            linewidth=1.4, linestyle="--", label="promedio sustitutos")
    ax.plot(serie["periodo"].cast(str), serie["tn_complementarios_prom"], color=SERIE[2],
            linewidth=1.4, linestyle="--", label="promedio complementarios")
    ax.set_xticks(ax.get_xticks()[::max(1, len(ax.get_xticks()) // 8)])
    limpiar(ax, "venta propia vs. promedio de vecinos, en el tiempo", "tn", "periodo")
    ax.legend(fontsize=7.5)
    fig.tight_layout()
    plt.show()


### Escalado: `tn0` crudo vs. `tn0_norm`

OJO con la lectura de este grafico: `B1` NO es un factor fijo por serie, se
recalcula fila a fila con la ventana propia de esa fila (`tn0..tn_lag{L}`, o
`tn_ma{ventana}` en `rolling_mean`). Es una normalizacion LOCAL/movil, no una
z-score global -- asi que `tn0_norm` no tiene por que verse "mas plano" en el
tiempo (un pico despues de una racha floja puede quedar MAS marcado, no menos,
porque `B1` venia bajo). Lo que sí tiene que cumplirse: `B1` mas grande para
los pares de mayor volumen (eso es lo que hace comparables series de escalas
distintas entre sí, que es el objetivo real del escalado).


In [ ]:
if "tn0_norm" in df.columns and "B1" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))

    pid_ej, cid_ej = None, None
    fila_vol = (df.group_by(KEYS).agg(pl.col("tn0").sum().alias("t")).sort("t", descending=True).head(1))
    filtro = [pl.col(k) == fila_vol[k][0] for k in KEYS]
    serie = df.filter(pl.all_horizontal(filtro)).sort("periodo")
    ax2 = axes[0].twinx()
    axes[0].plot(range(serie.height), serie["tn0"], color=TINTA, linewidth=1.6, label="tn0 (crudo)")
    ax2.plot(range(serie.height), serie["tn0_norm"], color=SERIE[0], linewidth=1.4,
             linestyle="--", label="tn0_norm (escalado)")
    limpiar(axes[0], f"{describir_producto(fila_vol['product_id'][0])[:35]}\ncrudo vs escalado", "tn0", "periodo (indice)")
    ax2.set_ylabel("tn0_norm")
    ax2.grid(False)

    b1_por_serie = df.group_by(KEYS).agg(pl.col("B1").mean().alias("B1"),
                                         pl.col("tn0").sum().alias("tn_total"))
    axes[1].scatter(b1_por_serie["tn_total"], b1_por_serie["B1"], s=5, alpha=.3, color=SERIE[1])
    axes[1].set_xscale("log")
    axes[1].set_yscale("log")
    limpiar(axes[1], "B1 (factor de escala) vs. volumen total del par\n(tiene que subir junto con el volumen)",
            "B1 (escala log)", "tn total del par (escala log)")
    fig.tight_layout()
    plt.show()

    print(f"\nB1: min {df['B1'].min():.3f}  mediana {df['B1'].median():.3f}  max {df['B1'].max():.3f}")
    print(f"tn0_norm: min {df['tn0_norm'].min():.3f}  mediana {df['tn0_norm'].median():.3f}  "
          f"max {df['tn0_norm'].max():.3f}")


### Checklist rapido


In [ ]:
print("Repasar a ojo:")
print("  [ ] las tarjetas de producto de arriba: cat1/cat2/cat3/brand tienen sentido comercial?")
print("  [ ] edad_producto == edad_cliente_producto solo para clientes que compraron desde el lanzamiento?")
print("  [ ] meses_sin_compra vuelve a 0 justo despues de una compra, no antes ni despues?")
print("  [ ] los productos de mayor peso_producto_acum son los que uno esperaria (marcas fuertes, categorias grandes)?")
print("  [ ] en la matriz de correlacion, los productos de la MISMA categoria tienden a quedar juntos (rojo)?")
print("  [ ] tn0_norm oscila en un rango CHICO Y PARECIDO entre pares de distinto volumen "
      "(no tiene que verse 'plano' en el tiempo: B1 es local/movil, no una z-score global)?")
print("  [ ] B1 crece con el volumen del par (log-log casi lineal)?")
